# 10 — Guardrails

**Module notebook — definitions only.**

Two validation nodes that sit at the edges of the graph: `guardrail_input` runs
before the Orchestrator and can short-circuit a bad request straight to `END`
(no agent runs); `guardrail_output` runs after whichever agent ran, and flags
(but doesn't discard) a low-quality result.

Both nodes only ever *add* to `state["errors"]` and set `state["blocked"]` —
they never raise exceptions, so a bad request ends the graph run cleanly
instead of crashing the kernel.

Depends on: nothing upstream — this module only reads/writes plain dict keys.

In [ ]:
SUPPORTED_AUDIO_TYPES = {"auto", "hinglish"}
SUPPORTED_LANGUAGES = {"english", "arabic"}


## Input guardrail

Checked before the Orchestrator ever runs, so it applies to *both* a
brand-new-recording request and a follow-up question:

- `audio_type`, `transcript_language`, and `summary_language` must each be
  one of the values the pipeline actually supports (three independent
  settings — see `09_agents.ipynb`).
- A new-recording request needs a non-empty `source`.
- A follow-up question needs non-empty text, *and* a transcript already in
  state (otherwise there's nothing for the RAG agent to search).

Note: `state["transcript"]` here refers to the **translated** transcript
(see `02_transcriber.ipynb`/`09_agents.ipynb`) — used only as a readiness
check ("has content_agent run successfully at all?"). The RAG agent itself
reads `state["raw_transcript"]`, which is always produced alongside it.

In [ ]:
def guardrail_input(state: dict) -> dict:
    old_errors = state.get("errors", [])
    errors = list(old_errors)
    blocked = False

    audio_type = (state.get("audio_type") or "auto").strip().lower()
    transcript_language = (state.get("transcript_language") or "english").strip().lower()
    summary_language = (state.get("summary_language") or "english").strip().lower()

    if audio_type not in SUPPORTED_AUDIO_TYPES:
        errors.append(f"Unsupported audio type \'{audio_type}\'. Supported: {sorted(SUPPORTED_AUDIO_TYPES)}.")
        blocked = True
    if transcript_language not in SUPPORTED_LANGUAGES:
        errors.append(f"Unsupported transcript language \'{transcript_language}\'. Supported: {sorted(SUPPORTED_LANGUAGES)}.")
        blocked = True
    if summary_language not in SUPPORTED_LANGUAGES:
        errors.append(f"Unsupported summary language \'{summary_language}\'. Supported: {sorted(SUPPORTED_LANGUAGES)}.")
        blocked = True

    has_transcript = bool(str(state.get("transcript", "")).strip())
    question = state.get("question")

    if question is not None:
        # A chat turn — will route to rag_agent.
        if not question.strip():
            errors.append("Empty question — nothing to answer.")
            blocked = True
        elif not has_transcript:
            errors.append("No transcript available yet — process a recording before asking questions.")
            blocked = True
    else:
        # A new-recording request — will route to content_agent.
        source = state.get("source")
        if not source or not str(source).strip():
            errors.append("No source provided — give a YouTube URL or a local file path.")
            blocked = True

    new_errors = errors[len(old_errors):]
    if new_errors:
        print("Guardrail (input) blocked this request:")
        for e in new_errors:
            print(f"  - {e}")

    return {
        **state,
        "audio_type": audio_type,
        "transcript_language": transcript_language,
        "summary_language": summary_language,
        "errors": errors,
        "blocked": blocked,
    }


## Output guardrail

Checked after whichever agent ran. Doesn't try to judge answer *quality* (that
needs an LLM-as-judge, which is a reasonable later upgrade) — for now it
catches the cheap, obvious failure: the agent silently returned nothing.

In [ ]:
def guardrail_output(state: dict) -> dict:
    old_errors = state.get("errors", [])
    errors = list(old_errors)
    blocked = state.get("blocked", False)

    if state.get("question") is not None:
        if not str(state.get("answer", "")).strip():
            errors.append("RAG agent returned an empty answer.")
            blocked = True
    else:
        if not str(state.get("transcript", "")).strip():
            errors.append("Content agent produced an empty transcript — check the audio source.")
            blocked = True
        if not str(state.get("summary", "")).strip():
            errors.append("Content agent produced an empty summary.")
            blocked = True

    new_errors = errors[len(old_errors):]
    if new_errors:
        print("Guardrail (output) flagged an issue:")
        for e in new_errors:
            print(f"  - {e}")

    return {**state, "errors": errors, "blocked": blocked}
